## LangChain components

### Install libraries

In [ ]:
!pip install langchain-community

Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [ ]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

### Configure .env and model

In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

False

### PromptTemplate
LangChain's ChatPromptTemplate is a template for building structured prompts for chat models. It allows us to define roles (system, user, assistant), variables, and message formats to generate LLM prompts in a consistent and reusable manner.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {domain}. Answer concisely."),
    ("human", "{question}"),
])

# Format the prompt (returns a list of messages)
messages = prompt.format_messages(
    domain="Python programming",
    question="What is a picking error?"
)
#print(messages)
llm = make_llm() 

response = llm.invoke(messages)
print(response.content)

### OutputParsers
Output parsers: turning loose text into structured data. Models naturally return plain text; applications often need structured data. This is where output parsers come in. Their job is to take the raw model response and:
1) Enforce some expected format (e.g. JSON, list, enum),
2) Parse it into a Python object or pydantic model
3) Fail loudly when the response doesn’t match the schema.

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

llm = make_llm()

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Keep your answers concise."),
    ("user", "Summarize in 1 sentence: {text}"),
])
parser = StrOutputParser()

# Manual steps, no LCEL
messages = prompt.format_messages(text="LangChain is a library for building LLM applications.")
response = llm.invoke(messages)
result = parser.parse(response.content)
# Without parser result would be identical, but real value comes with chains, not in manual parsing
print(result)

LangChain is a library designed to facilitate the development of applications using large language models (LLMs).


### Outputparsers with Pydantic 
A more interesting case is a structured parser, where the parser actually does work. Here's a PydanticOutputParser: Here parser.get_format_instructions() injects formatting rules into the prompt telling the model to emit JSON, and parser.parse() validates that JSON against the Pydantic model and returns a typed object.

In [9]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_openai import ChatOpenAI

class Person(BaseModel):
    name: str = Field(description="the person's full name")
    age: int = Field(description="the person's age in years")

llm = make_llm()
parser = PydanticOutputParser(pydantic_object=Person)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract the person info.\n{format_instructions}"),
    ("user", "{text}"),
])

# Manual steps
messages = prompt.format_messages(
    text="Anna is 34 years old.",
    format_instructions=parser.get_format_instructions(),
)
response = llm.invoke(messages)
person = parser.parse(response.content)

print(person)        # name='Anna' age=34
print(person.age)    # 34

name='Anna' age=34
34


### Chains
A "chain" in LangChain is exactly what the name suggests: a sequence of components wired together so that the output of one becomes the input of the next. The classic chain is the one we have already been building piece by piece : prompt → model → parser. Each link does one job, and the chain runs them in order.

In [18]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = make_llm()

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise assistant."),
    ("human", "{question}"),
])

parser = StrOutputParser()

# LCEL chain
chain = prompt | llm | parser

# Invoke
result = chain.invoke({"question": "What is LCEL?"})
print(result)  # Plain string
print(type(result))

LCEL can refer to different things depending on the context. One common interpretation is "Low-Cost Energy Lab," which focuses on developing affordable energy solutions. It could also refer to specific organizations, technologies, or concepts in various fields. If you have a specific context in mind, please provide more details for a more accurate explanation.
<class 'langchain_core.messages.base.TextAccessor'>


### Chains JSONParser

In [22]:
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI 

llm = make_llm()

class Person(BaseModel):
    name: str = Field(description="Person's full name")
    age: int = Field(description="Person's age")

parser = JsonOutputParser(pydantic_object=Person)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract structured data. {format_instructions}"),
    ("human", "{text}"),
]).partial(format_instructions=parser.get_format_instructions())

chain = prompt | llm | parser
result = chain.invoke({"text": "Einstein is 55 years old."})
print(result)  # {'name': 'Einstein', 'age': 55}
print(result["name"])
print(result["age"])
# print(result.name)
# print(result.age)

{'name': 'Einstein', 'age': 55}
Einstein
55


### Chains Comma Separated List

In [24]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI  

llm = make_llm()  

parser = CommaSeparatedListOutputParser()

prompt = ChatPromptTemplate.from_template(
    "List 5 {topic}. {format_instructions}"
).partial(format_instructions=parser.get_format_instructions())

chain = prompt | llm | parser
result = chain.invoke({"topic": "Python web frameworks"})
print(result)  # ['Django', 'Flask', 'FastAPI', 'Tornado', 'Starlette']

['Django', 'Flask', 'FastAPI', 'Pyramid', 'Tornado']


### ResponseSchema and OutputParser
LangChain's ResponseSchema defines the expected response format (e.g., JSON fields with name, type, and description) and helps the model return data in a predictable structure. The OutputParser interprets the raw LLM response and transforms it into the desired form (e.g., Pydantic object, JSON, list), facilitating further use within the application.

In [4]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import List

class TextAnalysis(BaseModel):
    tags: List[str] = Field(description="What are best tags describing text. Give maximum 5 tags separated by comma.")
    topic: str = Field(description="What is the topic of text. Use maximum couple of words.")

parser = JsonOutputParser(pydantic_object=TextAnalysis)
instructions = parser.get_format_instructions()

template = """
Specify tags and the main topic of the text. Return only JSON without markdown tags.
tags: What are best tags describing text. Give maximum 5 tags separated by comma.
topic: What is the topic of text. Use maximum couple of words

Format response as JSON as below:
'tags': ['sometag', 'othertag', 'anothertag', 'tag4', 'tag5']
'subject': 'Some subject of text'

text: {input}

{instructions}
"""
prompt = ChatPromptTemplate.from_template(template=template)

llm = make_llm()
chain = prompt | llm | parser
output_dict = chain.invoke({
    "input":"They picked a way among the trees, and their ponies plodded along, carefully avoiding the many writhing and interlacing roots.  There was no undergrowth.  The ground was rising steadily, and as they went forward it seemed that the trees became taller, darker, and thicker. There was no sound, except an occasional drip of moisture falling through the still leaves.  For the moment there was no whispering or movement among the branches; but they all got an uncomfortable feeling that they were being watched with disapproval, deepening to dislike and even enmity.  The feeling steadily grew, until they found themselves looking up quickly, or glancing back over their shoulders, as if they expected a sudden blow.",
    "instructions":instructions})
print(output_dict)


{'tags': ['trees', 'ponies', 'forest', 'feeling', 'watched'], 'topic': 'Exploring forest'}


### Runnables 
A Runnable is the common interface that every building block in a LangChain chain implements. It's the reason the pipe operator 
works at all. When we write prompt | llm | parser, each of those three is a Runnable, and the | connects them because they all 
speak the same protocol. So "Runnables in chains" really means: the standardized units that chains are made of.
RunnableLambda wraps an ordinary Python function so it becomes a Runnable we can drop into a chain. Any time 
we need a bit of custom transformation between steps, reshaping a dict, cleaning a string, we wrap it in RunnableLambda 
(or in many cases just pass the bare function and LangChain coerces it):

In [ ]:
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI  

llm = make_llm()  

def add_context(question: str) -> dict:
    return {
        "question": question,
        "context": "LangChain is a framework for LLM apps.",
    }

chain = (
    RunnableLambda(add_context)
    | ChatPromptTemplate.from_messages([
        ("system", "Use this context: {context}"),
        ("human", "{question}"),
    ])
    | llm
    | StrOutputParser()
)

result = chain.invoke("What is a goods-in discrepancy?")
print(result)

### RunnableParallel 
RunnableParallel (often written as a plain dict in a chain) runs several Runnables on the same input at once and collects their 
outputs into a dictionary. 

In [29]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

llm = make_llm()
parser = StrOutputParser()

# ── English chain ──────────────────────────────────────────────────
prompt_english = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer concisely in English."),
    ("human", "{question}")
])
chain_english = prompt_english | llm | parser

# ── French chain ───────────────────────────────────────────────────
prompt_french = ChatPromptTemplate.from_messages([
    ("system", "Tu es un assistant utile. Réponds brièvement en français."),
    ("human", "{question}")
])
chain_french = prompt_french | llm | parser

# ── Parallel runner ────────────────────────────────────────────────
parallel = RunnableParallel(
    english=chain_english,
    french=chain_french,
)

result = parallel.invoke({"question": "What is AI?"})

print("English:", result["english"])
print("French: ", result["french"])

English: AI, or artificial intelligence, refers to the simulation of human intelligence in machines programmed to think and learn like humans. It encompasses various technologies, including machine learning, natural language processing, and robotics, enabling systems to perform tasks that typically require human intelligence, such as problem-solving, understanding language, and recognizing patterns.
French:  L'IA, ou intelligence artificielle, désigne des systèmes informatiques capables d'effectuer des tâches qui nécessitent normalement l'intelligence humaine, comme la compréhension du langage, la reconnaissance d'images, et la prise de décision.
